# Clasificación de Parkinson

**Objetivo:** construir modelos de clasificación para predecir la variable `Diagnosis` a partir de las 32 variables clínicas comprendidas entre `Age` y `Constipation`.

Se comparan Decision Tree, SVC, Random Forest y MLP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar la base de datos
df = pd.read_csv("parkinsons_disease_data.csv")

print("Dimensiones:", df.shape)
print(df.head())
print("\nColumnas:")
print(df.columns.tolist())

In [ ]:
# Separación de variables predictoras y etiqueta
# En el CSV la columna aparece como 'Diagnosis' (D mayúscula).
X = df.loc[:, 'Age':'Constipation']
y = df['Diagnosis']

print("Número de variables predictoras:", X.shape[1])
print("Distribución de la variable objetivo:")
print(y.value_counts())
print("\nValores faltantes en X:", X.isna().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(classification_report(y_test, y_pred, digits=4))

    results.append({"Clasificador": name, "Accuracy": acc})

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
print("\nComparación final:")
print(results_df)

In [ ]:
# Gráfica comparativa
plt.figure(figsize=(8,5))
plt.bar(results_df["Clasificador"], results_df["Accuracy"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.xlabel("Clasificador")
plt.title("Comparación de Accuracy")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión del mejor modelo
best_name = results_df.iloc[0]["Clasificador"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title(f"Matriz de confusión - {best_name}")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks([0,1], ["No Parkinson", "Parkinson"])
plt.yticks([0,1], ["No Parkinson", "Parkinson"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]), ha="center", va="center")

plt.tight_layout()
plt.show()

print("Mejor modelo:", best_name)
print(f"Accuracy: {accuracy_score(y_test,best_pred):.4f}")